In [ ]:
#importa as bibliotecas que serão usadas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats as st

# **Etapa 1: abrir os arquivos e analizar os dados.**

**Para iniciar o projeto, primeiro iremos ler e analizar cada informação dos dados do projeto.**

In [ ]:
#Lê os arquivos
datasets = pd.read_csv('telecom_dataset_new.csv')
clients = pd.read_csv('telecom_clients.csv')

In [ ]:
#imprime as 5 primeiras linhas, as informações e o volume geral (quantidade total de linhas e colunas) dos dois dataframes
print('='*50)
print('CINCO PRIMEIRAS LINHAS DOS DATASETS:')
print('='*50)
print(datasets.head())

print('='*50)
print('INFORMAÇÕES GERAIS DOS DATASETS:')
print('='*50)
print(datasets.info())

print('='*50)
print('VOLUME DOS DATASETS:')
print('='*50)
print(datasets.shape)

In [ ]:
print('='*50)
print('CINCO PRIMEIRAS LINHAS DOS CLIENTES:')
print('='*50)
print(clients.head())

print('='*50)
print('INFORMAÇÕES GERAIS DOS CLIENTES:')
print('='*50)
print(clients.info())

print('='*50)
print('VOLUME DOS CLIENTES:')
print('='*50)
print(clients.shape)

### Clonclusões:

**Após analizar ambos os dataframes, podemos chegar as seguintes conclusões:**

**O dataframe 'datasets' possui uma quantidade considerável de dados nulos na coluna 'operator_id', além de que as colunas 'user_id' e 'date' possuem tipos de dados incorretos. Como 'user_id' indica um id único de usuário dentro dos serviços da CallMeMaybe, faria mais sentido em ele ser um valor string (object) e não um valor de número interio (int64). Quanto a 'date', percebe-se que ele está classificado como uma string (object) porém, por ele ser uma data, faria mais sentido ele estar registrado como um valor datetime. Também há uma pequena quantidade de valores nulos na coluna 'internal', que, por ser representada por valores True e False, faria mais sentido reclassificarmos ela como um tipo de dado booleano (bool), isso será ajustado na hora de corrigir os valores nulos.**

**Quanto ao dataframe 'clients', nota-se que ele não possui nenhum valor nulo em nenhuma de suas colunas, o que já é algo bom, porém, as colunas 'user_id' e 'date_start' sofrem dos mesmos problemas de classificação das colunas de 'datasets', com 'user_id' sendo classificado como número inteiro e 'date_start' sendo classificado como string.**

**Com os dados iniciais analizados, iremos prosseguir para o pré-processamento de dados, onde além de resolver os problemas encontrados acima também iremos verificar a existencia de valores duplicados.**

**NOTA: Apesar dos dados em 'operator_id' estarem como ponto flutuante, resolvi mante-lo assim por enquanto ao invés de converte-lo para string pois assim será mais fácil de verificar os dados ausentes presentes nesta coluna.**

# **Etapa 2: Pré-processamento dos dados.**

**Primeiro, iremos vamos iniciar corrigindo os tipos de dados nas colunas descritas no passo anterior.**

In [ ]:
#Correção de colunas em 'datasets'

#Converte 'user_id' e para string
datasets['user_id'] = datasets['user_id'].astype(str)

#Converte 'date' em datetime
datasets['date'] = pd.to_datetime(datasets['date'])

#Imprime as informações das colunas de 'datasets' para verificar a correção
print(datasets.info())

In [ ]:
#Correção de colunas em 'clients'

#Converte 'user_id' de inteiro para string
clients['user_id'] = clients['user_id'].astype(str)

#Converte 'date_start' em datetime
clients['date_start'] = pd.to_datetime(clients['date_start'])

#Imprime as informações das colunas de 'clients' para verificar a correção
print(clients.info())

**Com os tipos de dados corrigidos, prosseguiremos para a correção das colunas vazias em 'operator_id' em 'datasets'.**

**A coluna 'operator_id' é de extrema importância, pois ela indica o número de identificador único de cada operador da CallMeMaybe.**

**Vamos primeiro verificar se as linhas que possuem esse dado ausente apresentam algo em comum.**

In [ ]:
#Encontra e imprime as linhas com valores ausentes em 'operator_id'
print(datasets[datasets['operator_id'].isna()])

**Percebe-se que aparentemente uma boa quantidade das chamadas com operator_id vazio são aquelas que foram perdidas (is_missed_call = True) e eram entrantes (direction = in).**

**Para confirmar essas informações, vamos filtrar os dados gerais em 'datasets' com base nessas informações e ver se obtemos o mesmo número de linhas (8172).**

In [ ]:
#Cria o dataframe 'nan_operators', que filtra os dodos em 'datasets' para separar apenas as linhas com 'operator_id' vazio, 'is_missed_call' = True
#e 'direction' = in
nan_operators = datasets[(datasets['operator_id'].isna()) & (datasets['is_missed_call'] == True) & (datasets['direction'] == 'in')]

print(nan_operators.head(15))
print(nan_operators.shape)

**Confirmamos aqui que, apesar de ser a maioria esmagadora, nem todas as chamadas perdidas entrantes estão com o id do operador nulo pois recebemos 7899 linhas ao invés de 1872, logo, vamos filtrar os dados novamente porém pelos valores opostos para ver se descobrimos alguma outra semelhança.**

In [ ]:
#Filtra 'datasets' pelos mesmos parâmetros, porém agora ele busca os valores diferentes de 'is_missed_call' = True e 'direction' = in
nan_operators_opposite = datasets[(datasets['operator_id'].isna()) & ~((datasets['is_missed_call'] == True) & (datasets['direction'] == 'in'))]

print(nan_operators_opposite.head(15))
print(nan_operators_opposite.shape)

**Podemos ver que aqui pudemos encontrar o resto das linhas com 'operator_id' nulo (7899 + 273 = 8172), esses dados porém não possuem uma padronização clara comparada com o primeiro grupo mais predominante de valores ausentes.**

### Análise.

**A análise de valores nulos deste dataframe nos revela o seguinte:**

**A pricipal causa dos nulos vem de chamadas entrantes (direction = in) e que foram perdidas, o que nos da uma ideia de que, por serem chamadas perdidas, logo, que nunca foram iniciadas, o sistema pode não ter registrado o operador responsável por ela.**

**Já o segundo grupo, o menor dos dois, não apresenta um padrão distinto que nem o do primeiro, temos tanto chamadas perdidas quanto atendidas. Por serem um número bem pequeno em comparação com a quantidade total de linhas (o dataframe possui 53902 linhas totais, onde apenas 273 são pertencentes ao segundo grupo de dados vazios) é justo de se assumir que a falta do id do operador possa se dar por algum problema interno do sistema que falhou em registrar o número.**

**Agora, os valores 'operator_id' do primeiro grupo serão preenchidos por 'missed_call' e os do segundo serão preenchidos por 'internal_error'.**

In [ ]:
#Cria o filtro de condições para separar os tipos de valores ausentes
missed_calls = ((datasets['operator_id'].isna()) & (datasets['is_missed_call'] == True) & (datasets['direction'] == 'in'))
internal_errors = ((datasets['operator_id'].isna()) & ~((datasets['is_missed_call'] == True) & (datasets['direction'] == 'in')))

#Aplica as substituições
datasets.loc[missed_calls, 'operator_id'] = 'missed_call'
datasets.loc[internal_errors, 'operator_id'] = 'internal_error'

#Imprime algumas linhas do dataframe para visualizar s mudanças assim como as informações gerais para verificar se ainda há valores ausentes
print(datasets.head(20))
print(datasets.info())

**Os valores nulos em 'operator_id' foram corrigidos com sucesso! Agora precisamos analizar os poucos valores nulos em 'internal' (valor que indica se foi uma chamada interna entre operadores de um mesmo cliente).**

**Vamos repetir o mesmo processo, agora analizando a coluna 'internal'.**

In [ ]:
#Encontra e imprime as linhas com valores ausentes em 'internal'
print(datasets[datasets['internal'].isna()])

**Como pode-se observar, há apenas 117 valores nulos totais na coluna 'internal', portanto, o melhor a fazer seria preenche-los com o valor 'False' e converter o tipo de dados de object para bool, já que a quantidade de valores nulos é muito baixa em comparação com o total de linhas, isso não irá impactar notavelmente os dados neste dataframe.**

In [ ]:
#Preenche os valores nulos em 'internal' com 'False' usando a função .fillna().
datasets['internal'] = datasets['internal'].fillna('False')

#Converte a coluna 'internal' de string (object) para boolean (bool)
datasets['internal'] = datasets['internal'].astype(bool)

#Imprime as informações de datasets com os dados em 'internal' corrigidos
print(datasets.info())

**Com isso, os valores ausentes em 'datasets' estão corrigidos, agora, iremos procurar e processar possíveis valores duplicados em ambos os dataframes.**

In [ ]:
#Verifica se há linhas duplicadas em ambos os conjuntos de dados
datasets_duplicates = datasets[datasets.duplicated()]
clients_duplicates = clients[clients.duplicated()]

#Soma as possíveis linhas duplicadas totais de ambos dataframes
datasets_duplicates_sum = datasets.duplicated().sum()
clients_duplicates_sum = clients.duplicated().sum()

#Imprime as possíveis linhas duplicadas dos dataframes e a sua soma
print('='*50)
print('LINHAS DUPLICADAS EM DATASETS:')
print('='*50)
print(datasets_duplicates.head(15))

print('')
print('='*50)
print('TOTAL DE LINHAS DUPLICADAS EM DATASETS:')
print('='*50)
print(datasets_duplicates_sum)

print('')

print('='*50)
print('LINHAS DUPLICADAS EM CLIENTS:')
print('='*50)
print(clients_duplicates.head(15))

print('')
print('='*50)
print('TOTAL DE LINHAS DUPLICADAS EM CLIENTS:')
print('='*50)
print(clients_duplicates_sum)

**Podemos ver que após o processo de separação das linhas repetidas percebemos que o datasets possui um total de 4900 linhas que se repetem pelo menos uma vez.**

**Agora, no dataframe 'clients', não tivemos nenhum resultado ao encontrar linhas duplicadas, ou seja, nenhuma linha deste dataframe se repete.**

**Vamos remover todas as cópias de datasets para assim darmos prosseguimento à análise de dados.**

In [ ]:
#Usa a função .drop_duplicates() para remover todas as linhas duplicadas em 'datasets'
datasets = datasets.drop_duplicates().reset_index(drop = True)

#Verifica novamente se há linhas copiadas
datasets_duplicates = datasets[datasets.duplicated()]
datasets_duplicates_sum = datasets.duplicated().sum()

#Imprime os dados de linhas duplicadas em datasets novamente para verificar se o drop_duplicates teve êxito
print('='*50)
print('LINHAS DUPLICADAS EM DATASETS:')
print('='*50)
print(datasets_duplicates.head(15))

print('')
print('='*50)
print('TOTAL DE LINHAS DUPLICADAS EM DATASETS:')
print('='*50)
print(datasets_duplicates_sum)

#Imprime as informações gerais do dataframe sem a linhas duplicadas
print('='*50)
print('INFORMAÇÕES GERAIS DE DATASETS:')
print('='*50)
print(datasets.info())

**Conseguimos remover as 4900 linhas duplicadas, evidente tanto pelo fato de agora termos uma resposta vazia quando tentamos imprimir valores duplicados em 'datasets' como pelo fato do dataframe que antes tinha 53902 linhas agora caiu para 49002 (53902 - 4900 = 49002).**

### Conclusão:

**Com os valores de ambos os dataframes pré-processados, podemos seguir para a análise de dados em busca de descobrir quais operadores são os mais e menos eficientes.**

# **Etapa 3: Análise de dados.**

**Agora, será iniciado a análise exploratória de dados com fim de encontrarmos operadores ineficientes.**

**De acordo com a CallMeMaybe, um operador é considerado ineficiente se:**

1. **Possui muitas chamadas recebidas perdidas (internas ou externas).**
2. **Apresenta tempo de espera prolongado nas chamadas recebidas.**
3. **No caso de operadores responsáveis por chamadas de saída, realiza poucas chamadas ativas.**

**Assim, nossa análise exploratória será feita com base nesses pontos.**

**Já que precisaremos analizar o tempo de espera nas chamadas recebidas, primeiro, vamos criar uma coluna que mostre isso.**

In [ ]:
#Cria a coluna 'wait_time' em 'datasets' que calcula a diferença entre a duração total de uma chamada pelo tempo da chamada caso ela foi atendida
datasets['wait_time'] = datasets['total_call_duration'] - datasets['call_duration']

In [ ]:
#Une os dois dataframes pela coluna 'user_id'
datasets_and_clients = datasets.merge(clients, on = 'user_id')
print(datasets_and_clients.head())

**Vamos criar um dataframe completo de agrega diversas métricas de cada operador.**

In [ ]:
#Cria o dataframe 'operator_performance_overall' que agrupa os IDs de operador pela agregação de diversas métricas 
#possíveis em 'datasets' para uma melhor análise de dados.
operator_performance_overall = datasets.groupby('operator_id').agg(

    #Conta a quantidade de ligações de cada operador
    total_calls = ('calls_count', 'count'),

    #Calcula a média de chamdas totais
    total_calls_mean = ('calls_count', 'mean'),

    #Conta a quantidade total de chamadas atendidas ('is_missed_call' = False)
    total_answered_calls = ('calls_count', lambda x: x[datasets.loc[x.index, 'is_missed_call'] == False].count()),
    
    #Conta a quantidade total de chamadas perdidas ('is_missed_call' = True)
    total_missed_calls = ('calls_count', lambda x: x[datasets.loc[x.index, 'is_missed_call'] == True].count()),

    #Conta a quantidade total de chamadas de entrada ('direction' = 'in')
    total_calls_in = ('calls_count', lambda x: x[datasets.loc[x.index, 'direction'] == 'in'].count()),

    #Calcula a média de chamadas de entrada do operador ('direction' = 'in')
    calls_in_mean = ('calls_count', lambda x: x[datasets.loc[x.index, 'direction'] == 'in'].mean()),

    #Calcula a média de tempo em chamadas de saída do operador
    call_time_in_mean = ('call_duration', lambda x: x[datasets.loc[x.index, 'direction'] == 'in'].mean()),

    #Conta a quantidade total de chamadas de saída ('direction' = 'out')
    total_calls_out = ('calls_count', lambda x: x[datasets.loc[x.index, 'direction'] == 'out'].count()),

    #Calcula a média de chamadas de saída do operador ('direction' = 'out')
    calls_out_mean = ('calls_count', lambda x: x[datasets.loc[x.index, 'direction'] == 'out'].mean()),

    #Calcula a média de tempo em chamadas de saída do operador
    call_time_out_mean = ('call_duration', lambda x: x[datasets.loc[x.index, 'direction'] == 'out'].mean()),
    
    #Calcula a média do tempo de espera ('wait_time')
    wait_time_mean = ('wait_time', 'mean'),

    #Calcula a média do tempo de espera ('wait_time') em ligações atendidas ('is_missed_call' = False)
    wait_time_answered_mean = ('wait_time', lambda x: x[datasets.loc[x.index, 'is_missed_call'] == False].mean()),

    #Calcula a média do tempo de espera ('wait_time') em ligações atendidas ('is_missed_call' = True)
    wait_time_missed_mean = ('wait_time', lambda x: x[datasets.loc[x.index, 'is_missed_call'] == True].mean()),

    #Calcula a média da duração das chamadas atendidas ('call_duration')
    call_time_mean = ('call_duration', 'mean'),

    #Calcula a média do tempo total de ligações ('total_call_duration')
    total_call_mean = ('total_call_duration', 'mean')

).reset_index()

#Cria a coluna 'answered_call_rate' que calcula a taxa de chamadas atendidas por operador
operator_performance_overall['answered_call_rate'] = (operator_performance_overall['total_answered_calls'] / operator_performance_overall['total_calls'])

#Cria a coluna 'missed_call_rate' que calcula a taxa de chamadas perdidas por operador
operator_performance_overall['missed_call_rate'] = (operator_performance_overall['total_missed_calls'] / operator_performance_overall['total_calls'])

#Ordena o dataframe pelo total de chamadas perdidas
operator_performance_overall = operator_performance_overall.sort_values(by = 'total_calls', ascending = False).reset_index(drop = True)

#Imprime as 15 primeiras linhas do resultado e o total de linhas (operadores)
print(operator_performance_overall.head(15))
print('='*50)
print('TOTAL DE OPERADORES:')
print('='*50)
print(operator_performance_overall.shape)

**(Usarei os dados de chamada dos clientes para outras análises futuras.)**

**Há um total de 1094 operadores no total incluindo os valores 'missed_call' e 'internal_error'.**

**Com as métricas calculadas, vamos agora analizar essas informações usando gráficos.**

**Vamos começar encontrando o nível de correlação entre cada coluna usando um mapa de calor.**

In [ ]:
#Filtra o dataframe dos valores 'missed_call' e 'internal_error' para conseguir criar um mapa de calor
operator_performance_no_null = operator_performance_overall[~operator_performance_overall['operator_id'].isin(['missed_call', 'internal_error'])]

#Calcula a matriz de correlação entre as colunas
corr_matrix = operator_performance_no_null.corr()

#Cria um mapa de calor para análise de correlação entre os valores das colunas
plt.figure(figsize = (15, 10))
sns.heatmap(corr_matrix, annot = True, fmt = '.2f')
plt.show()

### 1. Analizar operadores que possuem muitas chamadas perdidas.

**Vamos começar descobrindo quais operadores são aqueles que possuem muitas chamadas perdidas.**

**Antes de irmos atrás de dentificar operadores ineficientes nesta questão, seria interessante primeiro definirmos o que faz um operador ser considerado eficiente. Vamos primeiro criar um gráfico de dispersão que demonstre a relação que as métricas de taxas de chamadas atendidas tem com as colunas de chamadas totais perdidas.**

In [ ]:
#Filtra o gráfico para excluir o id de operador 'missed_call' pois este valor só existe para preencher valores vazios que seguiam um 
#padrão de chamadas perdias, sendo ele um outlier que só irá atrapalhar a vizualização geral do gráfico
performance_filtered = operator_performance_overall[~operator_performance_overall['operator_id'].isin(['missed_call'])]

#Cria um gráfico de dispersão que compara a taxa de chamadas atendidas pela quantidade total de chamadas perdidas
performance_filtered.plot(figsize = (15, 10), kind = 'scatter', x = 'answered_call_rate', y = 'total_missed_calls')
plt.title('Taxas de chamadas atendidas pelo total de chamadas perdidas', fontsize = 16)
plt.xlabel('Taxa de chamadas atendidas', fontsize = 12)
plt.ylabel('Total de chamadas perdidas', fontsize = 12)
plt.show()

**De acordo com o gráfico, podemos ver que a maioria dos pontos (operadores) se concentram por volta de uma taxa de chamadas atendiadas entre 0.6 a 0.7 (60 a 70% de todas as chamadas foram atendidas) e possuem menos de 40 chamadas perdidas, esses operadores podem ser considerados como operadores eficientes que cumprem suas tarefas.**

**Com os operadores eficientes em chamadas atendidas determinados, vamos agora criar outro gráfico de dispersão, porém agora comparando as métricas opostas (taxas de chamadas perdidas por total de chamadas atendidas).**

In [ ]:
#Cria um gráfico de dispersão que compara a taxa de chamadas perdidas pela quantidade total de chamadas atendidas
operator_performance_overall.plot(figsize = (15, 10), kind = 'scatter', x = 'missed_call_rate', y = 'total_answered_calls')
plt.title('Taxas de chamadas perdidas pelo total de chamadas atendidas', fontsize = 16)
plt.xlabel('Taxa de chamadas perdidas', fontsize = 12)
plt.ylabel('Total de chamadas respondidas', fontsize = 12)
plt.show()

**Analizando o gráfico, podemos notar que a grande parte dos operadores se concentra em uma taxa de chamadas perdidas por volta de 0.35 (35%) e um total de ligações atendidas por volta de 50.**

**Com isso, é seguro de se afirmar que operadores que tem valores acima do primeiro fator e abaixo do segundo sejam considerados ineficientes.**

**Vamos agora analizar os valores da taxa de chamadas perdidas por um gráfico de caixa, com o intuito de achar uma média exata para assim determinarmos a partir de qual taxa de chamadas perdidas um operador se torna ineficiente.**

In [ ]:
#Cria um gráfico de caixa que demonstra a taxa de chamadas perdidas por operador
operator_performance_overall['missed_call_rate'].plot(figsize = (15, 10), kind = 'box')
plt.title('Distribuição de taxas de chamadas perdidas por operadores', fontsize = 16)
plt.xticks([1], ['Operadores'])
plt.ylabel('Taxa de chamadas perdidas')
q1_line = operator_performance_overall['missed_call_rate'].quantile(0.25)
plt.axhline(q1_line, color = 'blue', linestyle = '--', label = f'Primeiro quartil (25%): {q1_line:.2f}')
#Adiciona uma linha de referência para o valor da mediana
median_line = operator_performance_overall['missed_call_rate'].median()
plt.axhline(median_line, color = 'green', linestyle = '--', label = f'Mediana (50%): {median_line:.2f}')
#Adiciona uma linha de referência para o terceiro quartil
q3_line = operator_performance_overall['missed_call_rate'].quantile(0.75)
plt.axhline(q3_line, color = 'yellow', linestyle = '--', label = f'Terceiro quartil (75%): {q3_line:.2f}')
#Adiciona uma linha para visualizarmos o limite do tempo máximo antes do valor se tornar um outlier
iqr = q3_line - q1_line #Calcula IQR
max_line = q3_line + (1.5 * iqr)
plt.axhline(max_line, color = 'red', linestyle = '--', label = f'Valor máximo (100%): {max_line:.2f}')
plt.legend()
plt.show()

**De acordo com a mediana mostrada no gráfico, podemos ver que metade dos operadores tem uma taxa de chamadas perdidas de 33%.**

**A partir do terceiro quartil (25% após a mediana) já apresenta valores preocupantes, esses 25% dos operadores possuem uma taxa de 33 a 47% de ligações perdidas, não é a maioria, mas já é um certo problema.**

**Após o fim do terceiro quartil, os valores apresentam um aumento de taxa amplamente disperso, indo de 47 até 98% antes do valor se tornar um outlier, se 98% de chamadas perdidas não é considerado um outlier, significa que há muitos operadores que possuem muito mais chamadas perdidas do que atendidas.**

**Como complemento, vamos também criar um gráfico de caixa que calcula a distribuição do total de chamadas perdidas por operador:**

In [ ]:
#Vamos usar performance_filtered novamente para excluir o valor de 'missed_call' e obter uma melhor visualização dos dados

#Cria um gráfico de caixa que demonstra a distribuição do total de chamadas perdidas por operador
performance_filtered['total_missed_calls'].plot(figsize = (15, 10), kind = 'box')
plt.title('Distribuição do total de chamadas perdidas por operadores', fontsize = 16)
plt.xticks([1], ['Operadores'])
plt.ylabel('Total de chamadas perdidas')
q1_line = performance_filtered['total_missed_calls'].quantile(0.25)
plt.axhline(q1_line, color = 'blue', linestyle = '--', label = f'Primeiro quartil (25%): {q1_line:.2f}')
#Adiciona uma linha de referência para o valor da mediana
median_line = performance_filtered['total_missed_calls'].median()
plt.axhline(median_line, color = 'green', linestyle = '--', label = f'Mediana (50%): {median_line:.2f}')
#Adiciona uma linha de referência para o terceiro quartil
q3_line = performance_filtered['total_missed_calls'].quantile(0.75)
plt.axhline(q3_line, color = 'yellow', linestyle = '--', label = f'Terceiro quartil (75%): {q3_line:.2f}')
#Adiciona uma linha para visualizarmos o limite do tempo máximo antes do valor se tornar um outlier
iqr = q3_line - q1_line #Calcula IQR
max_line = q3_line + (1.5 * iqr)
plt.axhline(max_line, color = 'red', linestyle = '--', label = f'Valor máximo (100%): {max_line:.2f}')
plt.legend()
plt.show()

**De acordo com a mediana mostrada no gráfico, podemos ver que metade dos operadores tem um total de 5 chamadas perdidas.**

**O Terceiro Quartil é 18. Isto significa que 25% dos operadores (entre a mediana e Q3) perderam entre 5 e 18 chamadas. O número 18 é o limite superior para 75% dos operadores.**

**Após o fim do terceiro quartil, o valor aumenta para um máximo de aproximadamente 43 ligações totais perdidas, sendo qualquer valor acima disso um outlier, esses os quais aparentam ser muitos de acordo com o gráfico.**

### Conclusão do item 1:

**Podemos concluir que operadores que possuem uma taxa de ligações perdidas iguais ou maiores do que 47% e tem mais de 40 ligações perdidas podem ser classificados como ineficientes.**

### 2. Analizar operadores que possuem tempos de espera prolongados em chamadas atendidas.

**Com isso feito, vamos explorar essas informações a fim de encontrar um padrão para tempos de espera prolongados.**

**Vamos começar primeiro explorando essas informações a partir de um gráfico de dispersão que compara o total das ligações atendidas por média de tempo de espera.**

In [ ]:
#Cria uma gráfico de dispersão que analiza a média de tempo de espera de ligações por ligações totais atendidas por cada operador
operator_performance_overall.plot(figsize = (15, 10), kind = 'scatter', x = 'wait_time_answered_mean', y = 'total_answered_calls')
plt.title('Tempo médio de espera por ligações atendidas totais por operadores', fontsize = 16)
plt.xlabel('Médias de tempo de espera de chamadas por cada operador', fontsize = 12)
plt.ylabel('Total de ligações atendidas', fontsize = 12)
plt.show()

**Olhando gráfico, podemos perceber que grande parte dos operadores (pontos) ficam concentrados mais para a esquerda do gráfico, porém não podemos ser muito conclusivos quanto aos dados pois possuímos muitos outliers atrapalhando a visualização, vamos criar um filtro que remova qualquer operadores que tenha uma média de tempo de espera em ligações atendidas que passe de 1000 segundos:**

In [ ]:
#Filtra valores extremamente altos (outliers) para melhor visualização de dados
operator_wait_filtered = operator_performance_overall[operator_performance_overall['wait_time_answered_mean'] < 1000]

#Cria uma gráfico de dispersão que analiza a média de tempo de espera de ligações por ligações totais atendidas por cada operador
operator_wait_filtered.plot(figsize = (15, 10), kind = 'scatter', x = 'wait_time_answered_mean', y = 'total_answered_calls')
plt.title('Tempo médio de espera por ligações atendidas totais por operadores', fontsize = 16)
plt.xlabel('Médias de tempo de espera de chamadas por cada operador', fontsize = 12)
plt.ylabel('Total de ligações atendidas', fontsize = 12)
plt.show()

**Olhando os resultados do gráfico, podemos perceber que quanto menor é a média de tempo de espera de um operador, maior é a quantidade total de ligações de ele atende, assim sendo mais eficiente para a CallMeMaybe.**

**Percebe-se que quanto mais demorado é a média de tempo de espera de um operador é, mais abaixo é a quantidade total de ligações que ele atendeu e vice versa.**

**Vamos analizar se o tempo de espera traz uma possível reação de causa e efeito, vamos criar um novo gráfico de dispersão, agora comparando o a média do tempo de espera com o total de chamadas feitas por operador.**

In [ ]:
#Cria uma gráfico de dispersão que analiza a média de tempo de espera de ligações por ligações totais atendidas por cada operador
operator_wait_filtered.plot(figsize = (15, 10), kind = 'scatter', x = 'wait_time_answered_mean', y = 'call_time_mean')
plt.title('Tempo médio de espera por ligações atendidas por tempo médio de ligação por operadores', fontsize = 16)
plt.xlabel('Médias de tempo de espera de chamadas por cada operador', fontsize = 12)
plt.ylabel('Médias de tempo de chamadas por cada operador', fontsize = 12)
plt.show()

**Olhando os valores podemos observar que novamente os operadores se concentram no canto inferior esquerdo, indicando que operadores que atendem com tempos de espera curtos não possuem ligações muito longas, indicando que os operadores concedem serviços rápidos e práticos.**

**Agora, o que é curioso, quanto mais um operador demora na espera de uma ligação, maior é a média de tempo que a ligação dura, indicando que o operadores mais distantes do canto inferior esquerdo possuem problemas de velocidade de atendimento no geral, pois eles não só demoram na hora de começar a atender um cliente (wait_time) como também na hora de efetuar os atendimentos (wait_time_mean), não só reduzindo o potencial de ligações que o operador pode fazer no geral mas também efetuando um serviço extremamente frustrante para o cliente.**

**Vamos agora criar um gráfico de caixa para olharmos melhor as médias do tempo de espera em chamadas atendidas, além da quantidade de outliers presentes.**

In [ ]:
#Cria um gráfico de caixa para vizualizarmos os outliers nas médias de tempo de espera nas ligações atendidas por operadores
operator_wait_filtered['wait_time_answered_mean'].plot(figsize = (15, 10), kind = 'box')
plt.title('Distribuição das médias de tempo de espera em chamadas atendidas por cada operador', fontsize = 16)
plt.xticks([1], ['Operadores'])
plt.ylabel('Média do tempo de espera', fontsize = 12)
#Adiciona uma linha de referência para o primeiro quartil
q1_line = operator_wait_filtered['wait_time_answered_mean'].quantile(0.25)
plt.axhline(q1_line, color = 'blue', linestyle = '--', label = f'Primeiro quartil (25%): {q1_line:.2f}')
#Adiciona uma linha de referência para o valor da mediana
median_line = operator_wait_filtered['wait_time_answered_mean'].median()
plt.axhline(median_line, color = 'green', linestyle = '--', label = f'Mediana (50%): {median_line:.2f}')
#Adiciona uma linha de referência para o terceiro quartil
q3_line = operator_wait_filtered['wait_time_answered_mean'].quantile(0.75)
plt.axhline(q3_line, color = 'yellow', linestyle = '--', label = f'Terceiro quartil (75%): {q3_line:.2f}')
#Adiciona uma linha para visualizarmos o limite do tempo máximo antes do valor se tornar um outlier
iqr = q3_line - q1_line #Calcula IQR
max_line = q3_line + (1.5 * iqr)
plt.axhline(max_line, color = 'red', linestyle = '--', label = f'Valor máximo (100%): {max_line:.2f}')
plt.legend()
plt.show()

**De acordo com a mediana mostrada no gráfico, metade dos operadores demoram aproximadamente 42 segundos para iniciar uma chamada.**

**O terceiro quartil vale aproximadamente 118 segundos. Isto significa que 25% dos operadores (entre a mediana e Q3) demoraram de 46 a 118 segundos para iniciar uma chamada.**

**Após o fim do terceiro quartil, o valor aumenta para um máximo de aproximadamente 263 segundos esperando uma ligação começar, mais de 4 minutos, sendo qualquer valor acima disso um outlier, esses os quais aparentam ser muitos de acordo com o gráfico.**

### Conclusão do item 2:

**Considerando as informações descobartas pelos gráficos, podemos concluir que um operador que tem um tempo médio de espera de mais de 118 segundos é um operador ineficiente, pois eles não só demoram demais na hora de prestarem um serviço, como apresentam menores atendimentos totais, afetando o potencial de lucro para a CallMeMaybe.**

### 3. Verificar operadores que realizam chamadas de saída que são baixamente ativas.

**Para iniciarmos essa parte da análise, primeiro precisamos definir o que faz uma chamada ser considerada ativa.**

**Vamos primeiro começar filtrando apenas operadores que só realizam chamadas de saída ('direction' = 'out').**

In [ ]:
#Filtra os dados para incluir apenas operadores que só realizam chamadas de saída (operadores que não 
#realizaram nenhuma chamada de entrada ('direction' = 'in'))
operator_performance_only_out = operator_performance_overall[operator_performance_overall['total_calls_in'] == 0].reset_index(drop = True)

#Imprime as 5 primeiras linhas e o total de operadores que só realizam chamadas de saída
print(operator_performance_only_out.head(5))
print('='*50)
print('TOTAL DE OPERADORES DE SAÍDA:')
print('='*50)
print(operator_performance_only_out.shape)

**De todos os 1094 operadores, apenas 338 são responsáveis apenas por chamadas de saída.**

**Agora, precisamos determinar um tempo mínimo que irá determinar se uma chamada é ativa ou não.**

**Vamos voltar a usar brevemente os dados brutos no dataframe 'datasets' criado no início da análise exploratória de dados e criar um histograma demonstrando a distribuição da duração das chamadas de saída.**

In [ ]:
#Filtra 'datasets' para incluir apenas chamadas de saída ('direction = 'out') que foram atendidas ('is_missed_call = False')
active_out_calls = datasets[
    (datasets_and_clients['direction'] == 'out') & 
    (datasets_and_clients['is_missed_call'] == False)
]

In [ ]:
#Conta a quantidade de valores em 'call_duration'
duration_count = active_out_calls['call_duration'].value_counts()

#Transforma 'duration_count' em um dataframe
df_duration_count = duration_count.reset_index()

#Ordena a ordem do dataframe pela tempo de chamada do menor para o maior
df_duration_count = df_duration_count.sort_values(by = 'call_duration', ascending = True).reset_index(drop = True)

#Imprime a contagem até 60 segundos
print(df_duration_count.head(60))

**Com esses dados, vamos criar um gráfico de caixa a fim de determinar outliers.**

In [ ]:
#Cria um gráfico de caixa para encontrar outliers no tempo de chamada a fim de filtra-los para realizarmos uma melhor análise
df_duration_count['call_duration'].plot(figsize = (15, 10), kind = 'box')
plt.title('Teste', fontsize = 16)
plt.xticks([1], ['Contagem de duração chamadas'])
plt.ylabel('Contagem', fontsize = 12)
#Adiciona uma linha de referência para o primeiro quartil
q1_line = df_duration_count['call_duration'].quantile(0.25)
plt.axhline(q1_line, color = 'blue', linestyle = '--', label = f'Primeiro quartil (25%): {q1_line:.2f}')
#Adiciona uma linha de referência para o valor da mediana
median_line = df_duration_count['call_duration'].median()
plt.axhline(median_line, color = 'green', linestyle = '--', label = f'Mediana (50%): {median_line:.2f}')
#Adiciona uma linha de referência para o terceiro quartil
q3_line = df_duration_count['call_duration'].quantile(0.75)
plt.axhline(q3_line, color = 'yellow', linestyle = '--', label = f'Terceiro quartil (75%): {q3_line:.2f}')
#Adiciona uma linha para visualizarmos o limite do tempo máximo antes do valor se tornar um outlier
iqr = q3_line - q1_line #Calcula IQR
max_line = q3_line + (1.5 * iqr)
plt.axhline(max_line, color = 'red', linestyle = '--', label = f'Valor máximo (100%): {max_line:.2f}')
plt.legend()
plt.show()

**Como podemos ver, os dados possuem muitos outliers, o que irá dificultar totalmente a determinção de valores ineficientes, para melhorarmos a situação, vamos filtrar os dados para termos no máximo uma duração mais comum.**

**Vamos filtrar 'df_duration_count' e 'operator_performance_only_out' para termos ligações que durem no máximo 10 minutos (600 segundos) e reimprimir o gráfico de caixa com os valores devidamente filtrados.**

In [ ]:
#Dertermina um tempo máximo de ligação (600 segundos)
max_call_limit = 600

#Usa o valor para filtrar os dois dataframes
df_duration_count_filtered = df_duration_count[df_duration_count['call_duration'] <= max_call_limit]
operator_performance_only_out_filtered = operator_performance_only_out[operator_performance_only_out['call_time_mean'] <= max_call_limit]

#Imprime os resultados
print(df_duration_count_filtered.head())
print(operator_performance_only_out_filtered.head())

**Com os dados filtrados, vamos agora refazer o gráfico de caixa.**

In [ ]:
#Cria um gráfico de caixa para encontrar outliers no tempo de chamada a fim de filtra-los para realizarmos uma melhor análise
df_duration_count_filtered['call_duration'].plot(figsize = (15, 10), kind = 'box')
plt.title('Teste', fontsize = 16)
plt.xticks([1], ['Duração chamadas'])
plt.ylabel('Contagem', fontsize = 12)
#Adiciona uma linha de referência para o primeiro quartil
q1_line = df_duration_count_filtered['call_duration'].quantile(0.25)
plt.axhline(q1_line, color = 'blue', linestyle = '--', label = f'Primeiro quartil (25%): {q1_line:.2f}')
#Adiciona uma linha de referência para o valor da mediana
median_line = df_duration_count_filtered['call_duration'].median()
plt.axhline(median_line, color = 'green', linestyle = '--', label = f'Mediana (50%): {median_line:.2f}')
#Adiciona uma linha de referência para o terceiro quartil
q3_line = df_duration_count_filtered['call_duration'].quantile(0.75)
plt.axhline(q3_line, color = 'yellow', linestyle = '--', label = f'Terceiro quartil (75%): {q3_line:.2f}')
#Adiciona uma linha para visualizarmos o limite do tempo máximo antes do valor se tornar um outlier
iqr = q3_line - q1_line #Calcula IQR
max_line = q3_line + (1.5 * iqr)
plt.axhline(max_line, color = 'red', linestyle = '--', label = f'Valor máximo (100%): {max_line:.2f}')
plt.legend()
plt.show()

**Agora que nossos dados possuem valores mais realistas, vamos agora fazer uso de um gráfico de barras para determinar se há um valor de tempo curto que aparece com bastante frequência.**

**Vamos considerar um limiar de segundos, onde uma chamada só sera considerada ativa se passar de 100 segundos (1 minuto e 40 segundos) pois seria um tempo o bastante para  representar a duração mínima necessária para uma breve introdução ou confirmação de serviços antes de uma desconexão não intencional vinda tanto de um cliente quanto do próprio operador ocorrer.**

In [ ]:
#Filtra os 101 menores valores de tempo de ligações em 'df_duration_count_filtered' (de 0 até 100 segundos)
df_duration_count_top_101 = df_duration_count_filtered.head(101).sort_values(by = 'call_duration', ascending = True).reset_index(drop = True)

#Cria um gráfico de barras com essas informações
plt.figure(figsize = (15, 10))
sns.barplot(data = df_duration_count_top_101, x = 'call_duration', y = 'count')
plt.title('Distribuição do tempo de duração de chamadas de saída atendidas', fontsize = 16)
plt.xlabel('Duração da chamada (segundos)', fontsize = 12)
plt.xticks(rotation = 90, fontsize = 7)
plt.ylabel('Contagem', fontsize = 12)
plt.show()

**Podemos ver que há bastante ligações de até 100 segundos, em especial há mais de 70 ligações que tiveram no máximo 7 segundos, a maior contagem entre estes valores.**

**Outros valores que chamam a atenção são 6, 13, 24 32, 37, 44, 46, 54, 60 e 81, que aparentam ser os tempos de ligação mais frquentes neste agrupamento.**

**É seguro assumir que operadores que possuem uma média de tempo de ligação entre esses valores pode ser considerado inativo.**

**Vamos agora criar um gráfico de dispersão com os dados métricos para comparar o tempo médio em chamadas de saída pelo total de ligações de saída realizadas.**

In [ ]:
#Cria uma gráfico de dispersão que analiza a média de tempo de ligações de saída por ligações totais atendidas por cada operador
operator_performance_only_out_filtered.plot(figsize = (15, 10), kind = 'scatter', x = 'call_time_out_mean', y = 'total_answered_calls')
plt.title('Tempo médio de chamadas de saída por chamadas totais atendidas', fontsize = 16)
plt.xlabel('Médias de tempo de chamadas de saída', fontsize = 12)
plt.ylabel('Total de ligações atendidas', fontsize = 12)
plt.show()

**Podemos perceber uma concentração de operadores que possuem médias abaixo de 100 segundos (1 minuto e 40) e possuem menos de 10 ligações, enquanto operadores com médias maiores realizam muito mais ligações.**

**Com isso, é seguro assumir que operadores que tenham uma média de tempo de ligação que não passe de 100 segundos seja considerado inativo.**

### Conclusão do item 3:

**Podemos concluir que operadores que operadores com uma média de tempo de ligações ativas menores do que 100 sejam considerados inativos, logo, ineficientes, pois estes além de não permanecerem por muito tempo em ligações, eles também não acabam efetuando uma quantidade considerável delas.**

### 4. Explorar os dados dos clientes.

**Além das análises exploratórias de dados que foram realizadas nos três itens, como um passo extra, vamos conferir os valores em 'clients' com o intuito de conferir se existem informações que pode ser úteis para se analizar na hora da realização dos testes de hipótese.**

**Vamos começar analizando os dados brutos em 'clients'.**

In [ ]:
#Imprime as 5 primeiras linhas em 'clients'
print(clients.head())
print(clients.info())

**Neste dataframe há 732 linhas, indicando que essa é a quantidade total de clientes que a CallMeMaybe possui, com três colunas, 'user_id' identifica o id de usuário, coluna esta que está presente em 'datasets' também, ou seja, será perfeita para uni-los.**

**'tariff_plan' mostra o plano tarifário que o usuário é membro, essa coluna parece ser a mais interessante de analisar.**

**'date_start' indica a data em que o usuário foi registrado nos serviços da CallMeMaybe.**

**Com isso, vamos conferir o total de usuários por cada plano.**

In [ ]:
#Conta cada id de usuário único por plano e imprime as primeiras 5 linhas
plans_by_total_users = clients.groupby('tariff_plan')['user_id'].nunique()
print(plans_by_total_users.head())

**Há no total três planos tarifários, A, B e C, com A tendo 76 usuários apenas, o menos popular dos três, B tendo 261 usuários, sendo ele o intermediário e C tendo 395 usuários, o mais popular**

**Para analizar de forma mais profunda esses planos, vamos uni-los com os dados em 'datasets' por 'user_id'.**

In [ ]:
#Une 'clients' e 'datasets' em 'user_id'
datasets_and_clients = datasets.merge(clients, on = 'user_id')
print(datasets_and_clients.head())

**Vamos analizar o total de chamadas e o total da quantidade de ligações atendidas e perdidas entre os três planos.**

In [ ]:
#Conta o total de ligações em cada plano tarifário
all_call_count = datasets_and_clients.groupby('tariff_plan')['calls_count'].count().reset_index()
all_call_count.columns = ['tariff_plan', 'total_calls']

#Conta o total de ligações atendidas e perdidas ('is_missed_call' = False/True) em cada plano tarifário
answered_missed_call_count = datasets_and_clients.groupby(['tariff_plan', 'is_missed_call'])['calls_count'].count().reset_index()

#Une os dataframes
tariff_status = answered_missed_call_count.merge(all_call_count, on = 'tariff_plan')
print(tariff_status.head(6))

**Podemos ver os seguintes detalhes:**

**A maioria das ligações vem do plano tarifário C (18209 no total), com 10040 delas atendidas e 8169 perdidas.**

**O plano B tem 17237 ligações no total, com 9519 atendidas e 7718 perdidas.**

**O plano A tem menos ligações, 13556 no total, 7990 foram atendidas e 5566 perdidas.**

**Vamos calcular a taxa de ligações atendidas e perdidas dos três planos.**

In [ ]:
#Cria dois dataframes que filtram apenas por ligações perdias e atendidas respectivamente
tariff_missed_status = tariff_status[tariff_status['is_missed_call'] == True].copy()
tariff_answered_status = tariff_status[tariff_status['is_missed_call'] == False].copy()

#Calcula as taxas de chamadas perdias e atendidas respectivamente
tariff_missed_status['missed_call_rate'] = ((tariff_missed_status['calls_count'] / tariff_missed_status['total_calls']) * 100).round(2)
tariff_answered_status['answered_call_rate'] = ((tariff_answered_status['calls_count'] / tariff_answered_status['total_calls']) *100).round(2)

#Imprime as taxas de chamadas atendidas e perdidas respectivamente
print(tariff_answered_status.head())
print(tariff_missed_status.head())

**Podemos notar os seguintes datealhes:**

**Apesar do plano A ser o menos popular, ele apresenta a maior taxa de chamadas respondidas (58.94%) e a menor de chamadas perdidas (41.06%).**

**Os planos B e C, possuem taxas de chamadas perdidas (44.78% e 44.86%) e atendidas (55.22% e 55.14%) similares, apesar de C possuir mais usuários do que B.**

**Isso talvez seja indício de que um desses planos possa estar sendo afetado por uma maior quantidade de operadores ineficientes.**

**Vamos usar esses detalhes na hora de criarmos os testes de hipótese.**

# CONCLUSÃO DA ANÁLISE EXPLORATÓRIA DE DADOS:

**Após a conclusão da análise dos três itens, podemos concluir que um operador ineficiente seria aquele que:**

1. **Possui uma taxa de chamadas perdidas igual ou maior que 47% ('missed_call_rate' >= 0.47)**
2. **Possui um total de 40 ou mais chamadas totais perdidas ('total_missed_calls' >= 40)**
3. **Tem um tempo de espera em média maior do que 118 segundos na hora de iniciar uma chamada atendida ('wait_time_answered_mean' > 118).**
4. **Tem uma média de tempo de chamada abaixo de 100 segundos em chamadas de saída ('direction' = 'out'), indicando um baixo grau de atividade ('call_time_out_mean' < 100).**

**Logo, podemos dar as seguintes regras que irão classificar se um operador é ou não eficiente:**

**REGRA 1: Um operador é ineficiente se sua taxa de chamadas perdidas for igual ou maior do que 47%.**

**REGRA 2: Um operador é ineficiente se seu volume total de chamadas perdidas for maior que 40.**

**REGRA 3: Um operador é ineficiente se o tempo médio de espera nas chamadas atendidas for maior que 118 segundos.**

**REGRA 4: Um operador é ineficiente se, em chamadas de saída, sua média de duração de chamadas atendidas for abaixo de 100.**

# **Etapa 4: Teste de hipóteses.**

**Com os resultados da análise exploratória concluída, vamos testar uma hipótese:**

**Com as regras que determinam o que faz um operador ser considerado ineficiente determinados, vamos verificar se há um plano tarifário que é mais ou menos afetado por esses operadores.**

## HIPÓTESE: Há uma proporção significativamente diferente de chamadas envolvendo operadores ineficientes entre os usuários dos três planos.

### HIPÓTESE NULA (H0): Não há diferença significativa na proporção de chamadas vindas de operadores ineficientes entre os três planos tarifários.

### HIPÓTESE ALTERNATIVA (H1): Há uma diferença significativa na proporção de chamadas vindas de operadores ineficientes entre os três planos tarifários.

#### O limiar de alfa será 5% (0.05)

**Para testar essa hipótese, vamos primeiro criar uma função que realiza um teste Z como forma de automatizar o processo.**

In [ ]:
#Cria a função 'run_z_test' que recebe o dataframe a ser analizado, dois planos tarifários e o limiar de alfa
def run_z_test(df, tariff_plan_1, tariff_plan_2, alpha):
    
    #Filtra os dados dos dois planos
    group1 = df[df['tariff_plan'] == tariff_plan_1]['is_inefficient']
    group2 = df[df['tariff_plan'] == tariff_plan_2]['is_inefficient']
    
    #Calcula o número de chamadas com operadores ineficientes (X) e o número de chamadas no geral (N)
    x1 = group1.sum()
    n1 = len(group1)
    x2 = group2.sum()
    n2 = len(group2)
    
    #Calcula as proporções amostrais
    proportions_1 = x1 / n1
    proportions_2 = x2 / n2
    
    #Calcula a proporção combinada (Pooled Proportion)
    proportions_pooled = (x1 + x2) / (n1 + n2)
    
    #Cálcula o erro de desvio padrão (standard error), valor-z (z-score) e valor-p (p-value)
    try:
        std_err = np.sqrt(proportions_pooled * (1 - proportions_pooled) * (1/n1 + 1/n2))
        z_score = (proportions_1 - proportions_2) / std_err
        
        p_value = 2 * (1 - st.norm.cdf(abs(z_score)))
    #Caso aconteça uma divisão por zero
    except ZeroDivisionError:
        z_score, p_value = np.nan, np.nan
        
    reject_h0 = p_value < alpha
    
    #Cria um if-else que determina que se o valor-p se mostrar menor que alpha (reject_h0 = True), 
    #vamos rejeitar a hipótese nula, caso contrário (reject_h0 = False), não poderemos rejeitá-la.
    if reject_h0:
        conclusion = 'Rejeitamos a hipótese nula, pois há uma diferença significativamente diferente na quantidade de chamadas vindas de operadores ineficientes entre os usuários dos planos analizados.' 
    else:
        conclusion = 'Não podemos rejeitar a hipótese nula, pois a quantidade de chamadas vindas de operadores ineficientes entre os dois planos tarifários é muito próxima'
    
    print(f'\nComparação entre os planos {tariff_plan_1} e {tariff_plan_2}')
    print(f'Proporção de Ineficiência em {tariff_plan_1}: {proportions_1:.2f}% ({x1} de {n1})')
    print(f'Proporção de Ineficiência em {tariff_plan_2}: {proportions_2:.2f}% ({x2} de {n2})')
    print(f'Valor-z (Z-score): {z_score:.4f}')
    print(f'Valor-p (P-Value): {p_value:.4f}')
    print(f'Limiar de alfa: {alpha}')
    print(f'Conclusão: {conclusion}')
    return z_score, p_value, reject_h0

**Vamos agora filtrar os dataframes para termos apenas operadores que sejam considerados ineficientes.**

In [ ]:
#Cria 'inefficient_operators' que filtra 'operator_performance_overall' para termos apenas 
#operadores que batem com as 4 regras que foram determinadas para identificar um operador ineficiente
inefficient_operators = operator_performance_overall[(operator_performance_overall['missed_call_rate'] >= 0.47) |
                                                     (operator_performance_overall['total_missed_calls'] >= 40) |
                                                     (operator_performance_overall['wait_time_answered_mean'] > 118) |
                                                     (operator_performance_overall['call_time_out_mean'] < 100)]

#Cria uma lista que contém os ids dos operadores ineficientes
inefficient_operators_id_list = inefficient_operators['operator_id'].unique().tolist()

**E adicionar uma coluna booleana que indica quais operadores são ou não ineficientes.**

In [ ]:
#Cria o dataframe 'full_list' que copia o dataframe 'datasets_and_clients'
full_list = datasets_and_clients.copy()

#Dentro de 'full_list', cria a coluna 'is_inefficient', uma variável booleana que determina se o operador responsável por
#tal ligação faz ou não parte da lista 'inefficient_operators_id_list'
full_list['is_inefficient'] = full_list['operator_id'].isin(inefficient_operators_id_list)

In [ ]:
print('='*50)
print('---------------INÍCIO DOS TESTES Z---------------')
print('='*50)

run_z_test(full_list, 'A', 'B', 0.05)
run_z_test(full_list, 'A', 'C', 0.05)
run_z_test(full_list, 'B', 'C', 0.05)

**Conclusão:**

**Podemos ver que há, de fato, uma diferença estatística significante de total de chamadas de operadores ineficientes entre os três testes, a proporção em usuários dos planos tarifários de A e B são muito similares, mas as coisas mudam no plano C, que possui uma proporção significativamente maior do que A e B, sendo esse o plano tarifário mais afetado pelos operadores ineficientes.**

# **CONCLUSÃO:**

**Após a devida análise do dataframe, podemos concluir que operadores ineficientes possuem taxa de chamadas perdidas for igual ou maior a 47%, possuem um total de chamadas perdidas maior que 40, tem um tempo médio de espera nas chamadas atendidas maior que 118 segundos e tem uma média de duração de chamadas atendidas abaixo de 100.**

**Além disso, apesar de estarem presentes em todos os três planos tarifários, os operadores ineficientes aparentam afetar mais usuários do C.**

**Para a CallMeMaybe cuidar devidamente de operadores ineficientes, eles devem focar nos pontos que foram apresentados nesta análise.**

# LINK DE DASHBOARD NO TABLEAU PUBLIC:



LINK: https://public.tableau.com/app/profile/andr.trolesi/viz/dashboard_caso_principal/Painel1?publish=yes